# SAM-WM — Kaggle train → validation → final test → zero-shot OOD

Registered runtime: CPython 3.12. Every subprocess is launched with `sys.executable`, so the notebook cannot silently mix pyenv/conda interpreters. Use a GPU runtime if available. This notebook never tunes on the Freiburg final test or OOD labels. Run top-to-bottom. FAIRUrbTemp must be attached as an extracted Kaggle Dataset because its BORIS distribution does not expose one stable file URL in this repository.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil

# If uploaded as a Kaggle Dataset, locate the repository bundle.
candidates = list(Path('/kaggle/input').rglob('pyproject.toml'))
repo = next((p.parent for p in candidates if (p.parent/'src/coolworld/samwm.py').exists()), None)
if repo is None:
    repo = Path('/kaggle/working/SAM-WM')
    raise RuntimeError('Attach/extract the SAM-WM repository to /kaggle/input first.')
print('repo =', repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)],check=True)
os.chdir(repo)


In [ ]:
# Software/invariant gate — must pass before data acquisition/training.
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)


In [ ]:
# Real primary benchmark preflight. This downloads only official Zenodo files and verifies MD5.
from coolworld.benchmarks import load_freiburg, save_manifest
ds = load_freiburg('data/freiburg', k=4, download_if_missing=True)
print(ds.name, len(ds.timestamps), len(ds.station_ids), ds.temperature.shape, 'observed fraction=', ds.observed_mask.mean())
save_manifest(ds, '/kaggle/working/freiburg_manifest.json')


In [ ]:
# Train 3 frozen seeds. Do not select the best seed afterwards.
for seed in [0,1,2]:
    subprocess.run([sys.executable,'train.py','--config','config/train.yaml','--seed',str(seed),'--out','/kaggle/working/artifacts/freiburg'],check=True)


In [ ]:
# Freiburg held-out final test + zero-shot Novi Sad OOD-1 for every seed.
for seed in [0,1,2]:
    ck=f'/kaggle/working/artifacts/freiburg/seed_{seed}/best.pt'
    out=f'/kaggle/working/artifacts/eval/seed_{seed}'
    subprocess.run([sys.executable,'eval.py','--checkpoint',ck,'--data','freiburg','--out',out],check=True)
    subprocess.run([sys.executable,'eval.py','--checkpoint',ck,'--data','novisad','--out',out],check=True)


In [ ]:
# OOD-2: FAIRUrbTemp (Scientific Data 2026).
# Attach the official DOI 10.48620/93247 archive as an extracted Kaggle Dataset, then set this path.
fair_root = Path('/kaggle/input/fairurbtemp-extracted')
if fair_root.exists():
    for seed in [0,1,2]:
        ck=f'/kaggle/working/artifacts/freiburg/seed_{seed}/best.pt'
        out=f'/kaggle/working/artifacts/eval/seed_{seed}'
        subprocess.run([sys.executable,'eval.py','--checkpoint',ck,'--data','fairurbtemp','--root',str(fair_root),'--out',out],check=True)
else:
    print('FAIRUrbTemp not attached: OOD-2 intentionally NOT fabricated. Attach official extracted DOI 10.48620/93247 and rerun this cell.')


In [ ]:
# Aggregate, never cherry-pick.
subprocess.run([sys.executable,'summarize.py','--root','/kaggle/working/artifacts/eval','--out','/kaggle/working/artifacts/summary.json'],check=True)
print(Path('/kaggle/working/artifacts/summary.json').read_text())


In [ ]:
# Produce current figures from saved result artifacts.
# (Per-seed history curves can be plotted from artifacts/freiburg/seed_*/history.json.)
seed0=Path('/kaggle/working/artifacts/eval/seed_0')
args=[sys.executable,'plot.py',str(seed0/'freiburg_metrics.json'),str(seed0/'novisad_metrics.json'),'--out','/kaggle/working/artifacts/figures']
if (seed0/'fairurbtemp_metrics.json').exists(): args.insert(-2,str(seed0/'fairurbtemp_metrics.json'))
subprocess.run(args,check=True)


In [ ]:
# Optional real FortyGuard proof request. This consumes a real provider request only when you run it.
# Key remains a Kaggle Secret and is never printed.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['FORTYGUARD_API_KEY'] = UserSecretsClient().get_secret('FORTYGUARD_API_KEY')
    print('FortyGuard secret loaded into process environment (value hidden).')
except Exception as exc:
    print('No FortyGuard secret loaded:', type(exc).__name__)

# Run only after placing a real AOI GeoJSON at /kaggle/working/sanjose_aoi.geojson:
# subprocess.run([sys.executable,'fortyguard_check.py','--date','2026-08-26','--time','15:00','--aoi','/kaggle/working/sanjose_aoi.geojson','--granularity','100','--out','/kaggle/working/artifacts/fortyguard'],check=True)


In [ ]:
# Export reproducible artifacts for GitHub documentation / later UI integration.
archive = shutil.make_archive('/kaggle/working/SAM_WM_RESULTS','zip','/kaggle/working/artifacts')
print(archive)
